# Phase 0: Project Foundation, Environment Verification & Dataset Audit
## Framework: XAI-RiceGuard
**Project Title:** *A Lesion-Grounded Explainable and Uncertainty-Aware Deep Learning Framework for Robust Rice Leaf Blast and Brown Spot Detection*

---

> **RESEARCH INTEGRITY & PHASE 0 BOUNDARY RULE:**
> - **Phase 0 Objective:** Establish a scientifically rigorous, reproducible foundation, verify dataset integrity, audit YOLO annotations & RiceSeg ground truth pairings, and generate machine-readable manifests.
> - **Strict Boundary:** NO model training, NO data splitting, NO augmentation, NO preprocessing pipelines, NO XAI heatmaps, and NO calibration experiments are performed in this phase.
> - **Non-Destructive Audit:** All source images, masks, and metadata in Google Drive remain completely unaltered.

### Step 1: Computational Environment & Accelerator Detection
Dynamically probe the OS, Python runtime, PyTorch version, and GPU accelerator details (T4, V100, A100, or CPU fallback).

In [ ]:
import sys
import os
from pathlib import Path

# Ensure project root is in sys.path
PROJECT_ROOT = Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.phase0.environment import detect_environment, generate_environment_report

env_info = detect_environment()
env_report_text = generate_environment_report(env_info)
print(env_report_text)

### Step 2: Google Drive Mounting & Persistent Directory Verification
Mount Google Drive (when executing on Colab) and verify persistent artifact directories (`manifests`, `reports`, `checkpoints`, `experiments`, `logs`).

In [ ]:
# Mount Google Drive if running in Google Colab
if env_info['is_colab']:
    from google.colab import drive
    drive.mount('/content/drive')
    print("Google Drive successfully mounted at /content/drive")
else:
    print("Running locally: Google Drive mount skipped.")

from src.phase0.config import resolve_paths, load_project_config

paths = resolve_paths()
proj_config = load_project_config()

print(f"Active Environment: {paths['environment'].upper()}")
print(f"Configured Project Root: {paths['project_root']}")
print("Configured Dataset Roots:")
for name, root in paths["dataset_roots"].items():
    print(f"  - {name:<22}: {root}")

# Save environment snapshot into persistent reports directory
reports_dir = Path(paths["artifacts"]["reports"])
env_report_file = reports_dir / "phase0_environment_report.txt"
with open(env_report_file, "w", encoding="utf-8") as f:
    f.write(env_report_text)
print(f"\nSaved environment report to: {env_report_file}")

### Step 3: Dataset Discovery & File Inventory
Perform safe, non-destructive file discovery across all four configured dataset locations.

In [ ]:
from src.phase0.dataset_discovery import discover_dataset_structure

discovery = {}
for ds_key, root_path in paths["dataset_roots"].items():
    disc = discover_dataset_structure(root_path)
    discovery[ds_key] = disc
    print(f"[{ds_key.upper()}]")
    print(f"  Exists: {disc['exists']}")
    print(f"  Total Files: {disc.get('total_files', 0)} | Images: {disc.get('total_images', 0)} | Size: {disc.get('total_size_mb', 0)} MB")
    print(f"  Immediate Subdirs: {disc.get('immediate_subdirs', [])}")
    print("-" * 60)

### Step 4: Non-Destructive Image Integrity Audit & Manifest Generation
Inspect image headers, dimensions, channels, modes, corruption, and SHA-256 hashes for:
1. **RiceLeafDiseaseBD** (Primary Development Dataset)
2. **Sethy et al.** (External Generalization Dataset)
3. **RiceLeafDisease-BD5** (External Field Dataset)

In [ ]:
from src.phase0.image_audit import audit_images
from src.phase0.manifest_generator import generate_dataset_manifest

audit_results = {}
manifest_results = {}
manifests_dir = paths["artifacts"]["manifests"]

# 4.1 Primary Dataset Audit
p_disc = discovery.get("primary_dataset", {})
if p_disc.get("exists") and p_disc.get("total_images", 0) > 0:
    p_root = paths["dataset_roots"]["primary_dataset"]
    all_p_images = []
    for dp, _, fnames in os.walk(p_root):
        for fn in fnames:
            if Path(fn).suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp"}:
                all_p_images.append(os.path.relpath(os.path.join(dp, fn), p_root))
    
    def extract_p_class(rel_p):
        parts = Path(rel_p).parts
        if len(parts) >= 2:
            return parts[1]
        return parts[0]
    
    p_audit = audit_images("RiceLeafDiseaseBD", p_root, all_p_images, compute_hashes=True, class_extractor=extract_p_class)
    audit_results["primary_dataset"] = p_audit
    m_info = generate_dataset_manifest("RiceLeafDiseaseBD", "PRIMARY_DEVELOPMENT_DATASET", p_audit["image_records"], manifests_dir)
    manifest_results["RiceLeafDiseaseBD"] = m_info
    print(f"RiceLeafDiseaseBD: {p_audit['total_images']} images | Readable: {p_audit['readable_count']} | Unreadable: {p_audit['unreadable_count']}")
    print(f"  Classes: {p_audit['class_counts']}")

# 4.2 Sethy External Dataset Audit
s_disc = discovery.get("sethy_external", {})
if s_disc.get("exists") and s_disc.get("total_images", 0) > 0:
    s_root = paths["dataset_roots"]["sethy_external"]
    all_s_images = []
    for dp, _, fnames in os.walk(s_root):
        for fn in fnames:
            if Path(fn).suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp"}:
                all_s_images.append(os.path.relpath(os.path.join(dp, fn), s_root))
    
    s_audit = audit_images("Sethy_Rice_Leaf_Disease", s_root, all_s_images, compute_hashes=True)
    audit_results["sethy_external"] = s_audit
    m_info = generate_dataset_manifest("Sethy_Rice_Leaf_Disease", "EXTERNAL_DATASET", s_audit["image_records"], manifests_dir)
    manifest_results["Sethy_Rice_Leaf_Disease"] = m_info
    print(f"\nSethy External: {s_audit['total_images']} images | Readable: {s_audit['readable_count']} | Unreadable: {s_audit['unreadable_count']}")
    print(f"  Classes: {s_audit['class_counts']}")

# 4.3 BD5 External Field Dataset Audit
b_disc = discovery.get("bd5_external", {})
if b_disc.get("exists") and b_disc.get("total_images", 0) > 0:
    b_root = paths["dataset_roots"]["bd5_external"]
    all_b_images = []
    for dp, _, fnames in os.walk(b_root):
        for fn in fnames:
            if Path(fn).suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp"}:
                all_b_images.append(os.path.relpath(os.path.join(dp, fn), b_root))
    
    b_audit = audit_images("RiceLeafDisease_BD5", b_root, all_b_images, compute_hashes=True)
    audit_results["bd5_external"] = b_audit
    m_info = generate_dataset_manifest("RiceLeafDisease_BD5", "EXTERNAL_FIELD_DATASET", b_audit["image_records"], manifests_dir)
    manifest_results["RiceLeafDisease_BD5"] = m_info
    print(f"\nBD5 External: {b_audit['total_images']} images | Readable: {b_audit['readable_count']} | Unreadable: {b_audit['unreadable_count']}")
    print(f"  Classes: {b_audit['class_counts']}")

### Step 5: RiceLeafDiseaseBD YOLO Bounding Box & Metadata Audit
Formally audit YOLO `.txt` bounding box coordinates $[0, 1]$, syntax, class ID distributions, and correspondence with original images.

In [ ]:
from src.phase0.annotation_audit import audit_yolo_annotations, audit_dataset_metadata

p_root = paths["dataset_roots"]["primary_dataset"]
p_parent = str(Path(p_root).parent)

yolo_res = audit_yolo_annotations(p_root)
meta_res = audit_dataset_metadata(p_parent)

print("=" * 60)
print("RiceLeafDiseaseBD: YOLO Bounding Box Audit Results")
print("=" * 60)
print(f"Total YOLO Label Files (.txt): {yolo_res.get('total_label_files')}")
print(f"Total Rendered Visuals (.jpg): {yolo_res.get('total_visual_files')}")
print(f"Total Bounding Boxes: {yolo_res.get('total_bboxes')}")
print(f"YOLO Syntax Errors: {yolo_res.get('total_syntax_errors')}")
print(f"Coordinates Out of Bounds [0, 1]: {yolo_res.get('total_out_of_bounds')}")
print(f"Pairing with Originals: {yolo_res.get('pairing_with_originals')}")
print("\nDocumentation files found:")
for doc, info in meta_res.get('documentation_files', {}).items():
    print(f"  - {doc:<28}: Exists={info['exists']} ({info['size_bytes']} bytes)")

### Step 6: Sethy <-> RiceSeg-5932 Mask Pairing & Integrity Audit
Perform 1-to-1 stem matching, mask dimension alignment, and unique pixel value inspection between Sethy images and RiceSeg masks.

In [ ]:
from src.phase0.riceseg_audit import audit_sethy_riceseg_pairing

riceseg_res = audit_sethy_riceseg_pairing(
    paths["dataset_roots"]["sethy_external"],
    paths["dataset_roots"]["riceseg_ground_truth"]
)

p_sum = riceseg_res.get("pairing_summary", {})
print("=" * 60)
print("Sethy <-> RiceSeg-5932 Pairing Audit")
print("=" * 60)
print(f"Total Sethy Images: {p_sum.get('total_sethy_images')}")
print(f"Total RiceSeg Masks: {p_sum.get('total_riceseg_masks')}")
print(f"Exact Matched Pairs: {p_sum.get('exact_matches')}")
print(f"Unmatched Images: {p_sum.get('unmatched_images')}")
print(f"Unmatched Masks: {p_sum.get('unmatched_masks')}")
print(f"Dimension Mismatches: {p_sum.get('dimension_mismatches')}")
print("\nPer-Class Breakdown:")
for cls, cstat in p_sum.get("per_class_pairing", {}).items():
    print(f"  [{cls}] Images: {cstat['sethy_images']} | Masks: {cstat['riceseg_masks']} | Matched: {cstat['matched_pairs']}")

### Step 7: Duplicate & Cross-Dataset Leakage Audit
Inspect intra-dataset duplicate SHA-256 hashes and verify zero cross-contamination between Primary and External datasets.

In [ ]:
from src.phase0.duplicate_audit import audit_duplicates, audit_cross_dataset_leakage_risk

all_records = {ds_k: res["image_records"] for ds_k, res in audit_results.items()}
dup_counts = sum(audit_duplicates(recs)["total_duplicate_hashes"] for recs in all_records.values())
cross_dup = audit_cross_dataset_leakage_risk(all_records)

dup_summary = {
    "intra_dataset_duplicates_total": dup_counts,
    "cross_dataset_collisions_found": cross_dup.get("cross_dataset_collisions_found", False),
    "cross_dup_details": cross_dup
}

print(f"Intra-dataset duplicate hash groups found: {dup_counts}")
print(f"Cross-dataset collision risk detected: {cross_dup.get('cross_dataset_collisions_found')}")

### Step 8: Deterministic Sanity Visualizations (Fixed Seed: 42)
Display a small, controlled sample of images and image-mask pairs strictly for non-destructive visual verification.

In [ ]:
import random
from PIL import Image
import matplotlib.pyplot as plt

# Set deterministic seed for visual sample selection
random.seed(42)

# Sample from RiceLeafDiseaseBD
if "primary_dataset" in audit_results:
    p_records = audit_results["primary_dataset"]["image_records"]
    p_root = paths["dataset_roots"]["primary_dataset"]
    sample_recs = random.sample(p_records, min(4, len(p_records)))
    
    fig, axes = plt.subplots(1, len(sample_recs), figsize=(14, 4))
    for ax, rec in zip(axes, sample_recs):
        img_path = Path(p_root) / rec["relative_path"]
        with Image.open(str(img_path)) as img:
            ax.imshow(img)
            ax.set_title(f"{rec['class_name']}\n{rec['width']}x{rec['height']}", fontsize=9)
            ax.axis('off')
    plt.suptitle("Phase 0 Sanity Check: RiceLeafDiseaseBD Sample Images", fontsize=12)
    plt.tight_layout()
    plt.show()

# Sample Sethy Image + RiceSeg Mask Pair
sethy_root = Path(paths["dataset_roots"]["sethy_external"])
riceseg_root = Path(paths["dataset_roots"]["riceseg_ground_truth"])
sample_class = "Blast"
s_cls_dir = sethy_root / sample_class
r_cls_dir = riceseg_root / sample_class

if s_cls_dir.exists() and r_cls_dir.exists():
    s_imgs = sorted([f for f in s_cls_dir.iterdir() if f.suffix.lower() in {".jpg", ".png"}])
    if s_imgs:
        sample_img = s_imgs[0]
        sample_mask = r_cls_dir / sample_img.name
        if sample_mask.exists():
            fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(8, 4))
            with Image.open(str(sample_img)) as img, Image.open(str(sample_mask)) as msk:
                ax1.imshow(img)
                ax1.set_title(f"Sethy Image ({sample_class})")
                ax1.axis('off')
                ax2.imshow(msk, cmap='gray')
                ax2.set_title("RiceSeg Ground Truth Mask")
                ax2.axis('off')
            plt.suptitle("Phase 0 Sanity Check: Image-Mask Pairing Alignment", fontsize=12)
            plt.tight_layout()
            plt.show()

### Step 9: Generate IEEE-Ready Reports & Phase 0 Hard Gate Verification
Compile and output `reports/phase0_dataset_audit.md` and `reports/phase0_completion_report.md`.

In [ ]:
from src.phase0.report_generator import generate_phase0_audit_report, generate_phase0_completion_report

audit_report_path = reports_dir / "phase0_dataset_audit.md"
completion_report_path = reports_dir / "phase0_completion_report.md"

generate_phase0_audit_report(
    env_info=env_info,
    discovery_results=discovery,
    audit_results=audit_results,
    yolo_results=yolo_res,
    metadata_results=meta_res,
    riceseg_results=riceseg_res,
    duplicate_results=dup_summary,
    manifest_results=manifest_results,
    output_filepath=str(audit_report_path)
)
print(f"Comprehensive Dataset Audit Report generated at: {audit_report_path}")

audit_summary = {
    "primary_unreadable": audit_results.get("primary_dataset", {}).get("unreadable_count", 0),
    "sethy_unreadable": audit_results.get("sethy_external", {}).get("unreadable_count", 0),
    "riceseg_unmatched": p_sum.get("unmatched_images", 0),
    "bd5_unreadable": audit_results.get("bd5_external", {}).get("unreadable_count", 0),
}

comp_report = generate_phase0_completion_report(
    env_info=env_info,
    audit_summary=audit_summary,
    output_filepath=str(completion_report_path)
)
print(f"Phase 0 Completion Report generated at: {completion_report_path}")
print("\n" + "=" * 60)
print("PHASE 0 VALIDATION CHECKLIST & HARD GATE STATUS")
print("=" * 60)
print(comp_report)